# NB04b — Weighted CNN kondisi lampu

ResNet18 dengan bobot awal ImageNet menggabungkan embedding CNN, 77 fitur numerik (35 photometry bbox, 35 photometry foreground mask SAM2, 7 geometri mask), dan cluster e-means one-hot untuk membedakan `MobilStandar`, `MobilTidakStandar`, `MotorStandar`, `MotorTidakStandar`, dan `PantulanCahaya`. Crop dipertahankan seluruhnya, rasio aspeknya dijaga, sisi terpanjang diubah ke 224 px, lalu padding mean ImageNet ditambahkan agar tensor seragam tanpa center-crop atau distorsi. Pixel diubah ke rentang 0–1 lalu dinormalisasi dengan mean/std ImageNet. Loss weighted `CrossEntropyLoss` menimbang lima kelas berdasarkan distribusi train; angka kelas dan bobot loss dicetak serta disimpan ke `metrics.json`. Lima epoch awal melatih head saja sambil membekukan BatchNorm backbone, lalu backbone di-fine-tune dengan learning rate lebih kecil. Fitur numerik dihitung dari crop/mask bersih sebelum augmentasi dan diskalakan memakai mean/std train saja. Centroid e-means juga di-fit hanya pada train; jumlah cluster dipilih dengan BIC yang membandingkan objective linear dengan segmented-linear satu elbow interior pada k=2–12, tiga restart per k. Jika tidak ada elbow yang didukung, training berhenti dan objective tiap kandidat tetap tersimpan di `emeans_k_selection.csv`; kurva terpilih disimpan ke `emeans_k_selection.png`. Mask kosong memberi vektor foreground nol, dicatat dalam `mask_quality.json`, dan proporsi di atas 1% menghentikan training. Validation/test di-assign ke centroid tersimpan. Horizontal flip mencerminkan fitur centroid-x mask secara konsisten; ColorJitter hanya diterapkan pada citra CNN. NB02e harus selesai dengan manifest `complete` sebelum notebook ini dijalankan. Jika NB02c memiliki anotasi test yang sudah direview, metrik test dihitung terpisah. MSE/RMSE pada label multiclass adalah Brier score dan akar Brier score, bukan metrik bbox. Label hasil CNN menjadi kelas akhir per bbox; PantulanCahaya tetap dipelajari tetapi dikecualikan dari hitungan kendaraan dan overlay event.

In [ ]:
import os
import sys
from pathlib import Path

RISET_ROOT = Path(os.environ.get('RISET_PROJECT_ROOT', Path.cwd())).resolve()
sys.path.insert(0, str(RISET_ROOT))
sys.path.insert(0, str(RISET_ROOT / 'builders'))
from train_attribute_classifier import main

main()


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

history_path = RISET_ROOT / 'output' / 'attribute_classifier' / 'training_history.csv'
history_frame = pd.read_csv(history_path)
display(history_frame.tail())
display(Markdown('Plot lengkap tersimpan di `output/attribute_classifier/training_curves.png`. '
'Kolom test hanya ada bila test memiliki ground truth. Kurva pemilihan jumlah cluster ada di `emeans_k_selection.png`.'))
